# 02 - Ingeniería de Variables

## Predicción del riesgo de rotura de stock en un entorno logístico

Este notebook construye las variables necesarias para el modelo predictivo a partir del dataset original de ventas.  
El objetivo es generar variables temporales, variables de comportamiento histórico de la demanda y una variable objetivo binaria (`stockout_risk`) que represente el riesgo de rotura de stock.

> Nota metodológica: las variables `stock_estimated`, `safety_stock` y `future_demand_7d` se utilizan para construir la variable objetivo. Por tanto, **no deben utilizarse como variables predictoras en el entrenamiento del modelo**, ya que producirían leakage.


## 1. Carga de librerías y datos

Se carga el dataset original y se ordena por tienda, producto y fecha.  
Este orden es imprescindible para calcular correctamente lags, ventanas móviles y demanda futura por cada combinación `store_id` - `item_id`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

RAW_DATA_PATH = "../data/raw/retail_sales.csv"
PROCESSED_DATA_PATH = "../data/processed/stockout_dataset.csv"

df = pd.read_csv(RAW_DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["store_id", "item_id", "date"]
).reset_index(drop=True)

print("Shape inicial:", df.shape)
df.head()


Shape inicial: (4565000, 8)


,date,store_id,item_id,sales,price,promo,weekday,month
0,2019-01-01,store_1,item_1,41,21.30,0,1,1
1,2019-01-02,store_1,item_1,53,21.30,0,2,1
2,2019-01-03,store_1,item_1,39,21.30,0,3,1
3,2019-01-04,store_1,item_1,35,21.30,0,4,1
4,2019-01-05,store_1,item_1,51,17.04,1,5,1


## 2. Revisión inicial del dataset

Antes de crear nuevas variables se revisa la estructura básica del dataset, los tipos de datos y la existencia de nulos o duplicados.

In [2]:
display(df.info())

print("\nNulos por columna:")
display(df.isna().sum())

print("\nDuplicados:", df.duplicated().sum())

display(df.describe(include="all").T)

<class 'pandas.DataFrame'>
RangeIndex: 4565000 entries, 0 to 4564999
Data columns (total 8 columns):
 #   Column    Dtype         
---  ------    -----         
 0   date      datetime64[us]
 1   store_id  str           
 2   item_id   str           
 3   sales     int64         
 4   price     float64       
 5   promo     int64         
 6   weekday   int64         
 7   month     int64         
dtypes: datetime64[us](1), float64(1), int64(4), str(2)
memory usage: 278.6 MB


None


Nulos por columna:


date        0
store_id    0
item_id     0
sales       0
price       0
promo       0
weekday     0
month       0
dtype: int64


Duplicados: 0


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
date,4565000,NaN,NaN,NaN,2021-07-01 11:59:59.999999,2019-01-01 00:00:00,2020-04-01 00:00:00,2021-07-01 12:00:00,2022-10-01 00:00:00,2023-12-31 00:00:00,NaN
store_id,4565000,50,store_1,91300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item_id,4565000,50,item_1,91300,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sales,4565000.0,NaN,NaN,NaN,29.26466,0.0,18.0,27.0,38.0,139.0,15.009957
price,4565000.0,NaN,NaN,NaN,53.993231,8.02,31.97,53.52,75.36,99.99,25.784608
promo,4565000.0,NaN,NaN,NaN,0.099999,0.0,0.0,0.0,0.0,1.0,0.299998
weekday,4565000.0,NaN,NaN,NaN,3.001643,0.0,1.0,3.0,5.0,6.0,1.999315
month,4565000.0,NaN,NaN,NaN,6.523549,1.0,4.0,7.0,10.0,12.0,3.448534


## 3. Variables temporales

Se generan variables de calendario que pueden capturar patrones estacionales y ciclos de demanda:

- Año.
- Mes.
- Trimestre.
- Semana del año.
- Día del mes.
- Día de la semana.

Estas variables son relevantes porque la demanda puede variar según el momento del año, el mes o el día de la semana.

In [3]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.weekday

df[["date", "year", "month", "quarter", "week", "day", "weekday"]].head()

,date,year,month,quarter,week,day,weekday
0,2019-01-01,2019,1,1,1,1,1
1,2019-01-02,2019,1,1,1,2,2
2,2019-01-03,2019,1,1,1,3,3
3,2019-01-04,2019,1,1,1,4,4
4,2019-01-05,2019,1,1,1,5,5


## 4. Variables históricas de demanda: lags

Se crean variables de ventas pasadas para cada combinación tienda-producto:

- Ventas hace 7 días.
- Ventas hace 14 días.
- Ventas hace 30 días.

Estas variables permiten incorporar el comportamiento histórico reciente sin utilizar información futura.

In [4]:
group_cols = ["store_id", "item_id"]

df["sales_lag_7"] = (
    df.groupby(group_cols)["sales"]
      .shift(7)
)

df["sales_lag_14"] = (
    df.groupby(group_cols)["sales"]
      .shift(14)
)

df["sales_lag_30"] = (
    df.groupby(group_cols)["sales"]
      .shift(30)
)

df[["store_id", "item_id", "date", "sales", "sales_lag_7", "sales_lag_14", "sales_lag_30"]].head(35)


,store_id,item_id,date,sales,sales_lag_7,sales_lag_14,sales_lag_30
0,store_1,item_1,2019-01-01,41,NaN,NaN,NaN
1,store_1,item_1,2019-01-02,53,NaN,NaN,NaN
2,store_1,item_1,2019-01-03,39,NaN,NaN,NaN
3,store_1,item_1,2019-01-04,35,NaN,NaN,NaN
4,store_1,item_1,2019-01-05,51,NaN,NaN,NaN
5,store_1,item_1,2019-01-06,38,NaN,NaN,NaN
6,store_1,item_1,2019-01-07,45,NaN,NaN,NaN
7,store_1,item_1,2019-01-08,48,41.0,NaN,NaN
8,store_1,item_1,2019-01-09,50,53.0,NaN,NaN
9,store_1,item_1,2019-01-10,44,39.0,NaN,NaN


## 5. Variables de demanda mediante ventanas móviles

Se calculan medias móviles históricas de ventas para cada combinación tienda-producto.

Importante: se aplica `shift(1)` antes de calcular la ventana móvil para evitar que la venta del día actual entre en el cálculo de la media histórica del propio día.

In [5]:
df["rolling_mean_7"] = (
    df.groupby(group_cols)["sales"]
      .transform(lambda s: s.shift(1).rolling(window=7, min_periods=7).mean())
)

df["rolling_mean_14"] = (
    df.groupby(group_cols)["sales"]
      .transform(lambda s: s.shift(1).rolling(window=14, min_periods=14).mean())
)

df["rolling_mean_30"] = (
    df.groupby(group_cols)["sales"]
      .transform(lambda s: s.shift(1).rolling(window=30, min_periods=30).mean())
)

df["rolling_std_30"] = (
    df.groupby(group_cols)["sales"]
      .transform(lambda s: s.shift(1).rolling(window=30, min_periods=30).std())
)

df[[
    "store_id", "item_id", "date", "sales",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_30", "rolling_std_30"
]].head(35)

,store_id,item_id,date,sales,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_30
0,store_1,item_1,2019-01-01,41,NaN,NaN,NaN,NaN
1,store_1,item_1,2019-01-02,53,NaN,NaN,NaN,NaN
2,store_1,item_1,2019-01-03,39,NaN,NaN,NaN,NaN
3,store_1,item_1,2019-01-04,35,NaN,NaN,NaN,NaN
4,store_1,item_1,2019-01-05,51,NaN,NaN,NaN,NaN
5,store_1,item_1,2019-01-06,38,NaN,NaN,NaN,NaN
6,store_1,item_1,2019-01-07,45,NaN,NaN,NaN,NaN
7,store_1,item_1,2019-01-08,48,43.142857,NaN,NaN,NaN
8,store_1,item_1,2019-01-09,50,44.142857,NaN,NaN,NaN
9,store_1,item_1,2019-01-10,44,43.714286,NaN,NaN,NaN


## 6. Tendencia reciente de demanda

Se crea una variable de tendencia comparando la media de ventas de los últimos 7 días frente a la media de los últimos 30 días.

- Valor positivo: demanda reciente por encima de la media de 30 días.
- Valor negativo: demanda reciente por debajo de la media de 30 días.

In [6]:
df["trend_7_30"] = (
    df["rolling_mean_7"] -
    df["rolling_mean_30"]
)

df[["rolling_mean_7", "rolling_mean_30", "trend_7_30"]].describe()

,rolling_mean_7,rolling_mean_30,trend_7_30
count,4.547500e+06,4.490000e+06,4.490000e+06
mean,2.926327e+01,2.926500e+01,1.999428e-02
std,1.336282e+01,1.322533e+01,2.264629e+00
min,3.000000e+00,5.200000e+00,-1.491905e+01
25%,1.842857e+01,1.840000e+01,-1.390476e+00
50%,2.757143e+01,2.760000e+01,-3.809524e-02
75%,3.814286e+01,3.810000e+01,1.338095e+00
max,9.714286e+01,8.576667e+01,1.873333e+01


## 7. Variables logísticas simuladas

El dataset original no contiene stock real ni stock de seguridad.  
Para poder construir un caso de negocio de riesgo de rotura de stock, se generan variables simuladas:

- `stock_estimated`: estimación de stock disponible basada en la demanda media reciente.
- `safety_stock`: stock de seguridad calculado a partir de la variabilidad de demanda.
- `lead_time_days`: plazo de reposición simulado.

Estas variables permiten construir una aproximación del problema logístico, pero deben interpretarse como una simulación académica, no como datos reales de inventario.

In [7]:
np.random.seed(42)

coverage_days = np.random.randint(
    5,
    10,
    len(df)
)

df["stock_estimated"] = (
    df["rolling_mean_30"] * coverage_days
)

df["safety_stock"] = (
    df["rolling_std_30"] * 1.5
)

df["lead_time_days"] = np.random.randint(
    1,
    8,
    len(df)
)

df[["rolling_mean_30", "rolling_std_30", "stock_estimated", "safety_stock", "lead_time_days"]].describe()


,rolling_mean_30,rolling_std_30,stock_estimated,safety_stock,lead_time_days
count,4.490000e+06,4.490000e+06,4.490000e+06,4.490000e+06,4.565000e+06
mean,2.926500e+01,6.767278e+00,2.048796e+02,1.015092e+01,4.000577e+00
std,1.322533e+01,2.658027e+00,1.031637e+02,3.987040e+00,1.999433e+00
min,5.200000e+00,1.751518e+00,2.600000e+01,2.627277e+00,1.000000e+00
25%,1.840000e+01,4.694825e+00,1.245000e+02,7.042237e+00,2.000000e+00
50%,2.760000e+01,6.221949e+00,1.866000e+02,9.332923e+00,4.000000e+00
75%,3.810000e+01,8.302402e+00,2.661333e+02,1.245360e+01,6.000000e+00
max,8.576667e+01,2.491459e+01,7.581000e+02,3.737188e+01,7.000000e+00


## 8. Demanda futura a 7 días

Se calcula la demanda futura de los próximos 7 días para cada combinación tienda-producto.

Esta variable se usa **solo para construir el target**.  
No debe utilizarse como feature en el entrenamiento porque contiene información futura.

La fórmula se alinea para que, en cada fecha, represente la suma de ventas desde `t+1` hasta `t+7`.

In [8]:
df["future_demand_7d"] = (
    df.groupby(group_cols)["sales"]
      .transform(
          lambda s: s.shift(-1)
                     .rolling(window=7, min_periods=7)
                     .sum()
                     .shift(-6)
      )
)

df[["store_id", "item_id", "date", "sales", "future_demand_7d"]].head(15)

,store_id,item_id,date,sales,future_demand_7d
0,store_1,item_1,2019-01-01,41,309.0
1,store_1,item_1,2019-01-02,53,306.0
2,store_1,item_1,2019-01-03,39,311.0
3,store_1,item_1,2019-01-04,35,314.0
4,store_1,item_1,2019-01-05,51,293.0
5,store_1,item_1,2019-01-06,38,303.0
6,store_1,item_1,2019-01-07,45,298.0
7,store_1,item_1,2019-01-08,48,296.0
8,store_1,item_1,2019-01-09,50,293.0
9,store_1,item_1,2019-01-10,44,295.0


## 9. Construcción de la variable objetivo

La variable `stockout_risk` toma valor:

- `1`: existe riesgo de rotura de stock.
- `0`: no existe riesgo de rotura de stock.

Se considera que hay riesgo si la demanda futura de los próximos 7 días supera el stock estimado disponible tras considerar el stock de seguridad:

```text
future_demand_7d > stock_estimated - safety_stock
```

Esta definición convierte el problema en una tarea de clasificación binaria.

In [9]:
df["stockout_risk"] = np.where(
    df["future_demand_7d"] > (df["stock_estimated"] - df["safety_stock"]),
    1,
    0
)

df[["future_demand_7d", "stock_estimated", "safety_stock", "stockout_risk"]].head()

,future_demand_7d,stock_estimated,safety_stock,stockout_risk
0,309.0,NaN,NaN,0
1,306.0,NaN,NaN,0
2,311.0,NaN,NaN,0
3,314.0,NaN,NaN,0
4,293.0,NaN,NaN,0


## 10. Eliminación de nulos generados por ventanas temporales

Los lags, rolling windows y la demanda futura generan nulos al inicio y al final de cada serie tienda-producto.  
Estos registros se eliminan después de crear todas las variables.

In [10]:
print("Shape antes de eliminar nulos:", df.shape)

nulls_before = df.isna().sum()
display(nulls_before[nulls_before > 0])

df = df.dropna().reset_index(drop=True)

print("Shape después de eliminar nulos:", df.shape)

nulls_after = df.isna().sum()
display(nulls_after[nulls_after > 0])

Shape antes de eliminar nulos: (4565000, 25)


sales_lag_7         17500
sales_lag_14        35000
sales_lag_30        75000
rolling_mean_7      17500
rolling_mean_14     35000
rolling_mean_30     75000
rolling_std_30      75000
trend_7_30          75000
stock_estimated     75000
safety_stock        75000
future_demand_7d    17500
dtype: int64

Shape después de eliminar nulos: (4472500, 25)


Series([], dtype: int64)

## 11. Revisión de la distribución del target

Se revisa la proporción de clases para comprobar si el problema está equilibrado o si existe desbalance.

In [11]:
target_counts = df["stockout_risk"].value_counts().sort_index()
target_distribution = df["stockout_risk"].value_counts(normalize=True).sort_index()

display(pd.DataFrame({
    "count": target_counts,
    "proportion": target_distribution
}))

print("Proporción de riesgo de stockout:", round(target_distribution.get(1, 0), 4))

,count,proportion
stockout_risk,,
0,1883040,0.421026
1,2589460,0.578974


Proporción de riesgo de stockout: 0.579


## 12. Control de leakage

Se definen las variables que **no deben utilizarse como features** en el entrenamiento:

- `future_demand_7d`: información futura.
- `stockout_risk`: variable objetivo.
- `stock_estimated`: forma parte de la regla de construcción del target.
- `safety_stock`: forma parte de la regla de construcción del target.

Estas variables pueden conservarse en el dataset procesado para análisis, trazabilidad y métricas de negocio, pero deben excluirse en el notebook de modelado.

In [12]:
leakage_columns = [
    "future_demand_7d",
    "stockout_risk",
    "stock_estimated",
    "safety_stock"
]

model_features = [
    "sales",
    "price",
    "promo",
    "weekday",
    "month",
    "year",
    "quarter",
    "week",
    "day",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_30",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_std_30",
    "trend_7_30",
    "lead_time_days"
]

print("Variables excluidas por leakage:")
print(leakage_columns)

print("\nFeatures recomendadas para modelado:")
print(model_features)

missing_features = [col for col in model_features if col not in df.columns]
print("\nFeatures no encontradas:", missing_features)

Variables excluidas por leakage:
['future_demand_7d', 'stockout_risk', 'stock_estimated', 'safety_stock']

Features recomendadas para modelado:
['sales', 'price', 'promo', 'weekday', 'month', 'year', 'quarter', 'week', 'day', 'sales_lag_7', 'sales_lag_14', 'sales_lag_30', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_std_30', 'trend_7_30', 'lead_time_days']

Features no encontradas: []


## 13. Revisión final del dataset procesado

Se comprueba la estructura final del dataset antes de guardarlo.

In [13]:
print("Shape final:", df.shape)

display(df.head())
display(df.describe().T)

print("Columnas finales:")
print(df.columns.tolist())

Shape final: (4472500, 25)


,date,store_id,item_id,sales,price,promo,weekday,month,year,quarter,week,day,sales_lag_7,sales_lag_14,sales_lag_30,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_30,trend_7_30,stock_estimated,safety_stock,lead_time_days,future_demand_7d,stockout_risk
0,2019-01-31,store_1,item_1,50,21.30,0,3,1,2019,1,5,31,40.0,46.0,41.0,42.714286,42.142857,42.766667,6.251529,-0.052381,299.366667,9.377293,4,338.0,1
1,2019-02-01,store_1,item_1,41,21.30,0,4,2,2019,1,5,1,38.0,40.0,53.0,44.142857,42.428571,43.066667,6.378484,1.076190,344.533333,9.567726,1,336.0,1
2,2019-02-02,store_1,item_1,34,21.30,0,5,2,2019,1,5,2,34.0,33.0,39.0,44.571429,42.500000,42.666667,6.104455,1.904762,341.333333,9.156682,2,352.0,1
3,2019-02-03,store_1,item_1,34,21.30,0,6,2,2019,1,5,3,40.0,39.0,35.0,44.571429,42.571429,42.500000,6.273920,2.071429,212.500000,9.410880,1,375.0,1
4,2019-02-04,store_1,item_1,64,17.04,1,0,2,2019,1,6,4,44.0,38.0,51.0,43.714286,42.214286,42.466667,6.317645,1.247619,297.266667,9.476468,1,350.0,1


,count,mean,min,25%,50%,75%,max,std
date,4472500,2021-07-13 00:00:00.000001,2019-01-31 00:00:00,2020-04-22 00:00:00,2021-07-13 00:00:00,2022-10-03 00:00:00,2023-12-24 00:00:00,NaN
sales,4472500.0,29.273587,0.0,18.0,27.0,38.0,139.0,15.042329
price,4472500.0,53.993612,8.02,31.97,53.52,75.36,99.99,25.784637
promo,4472500.0,0.099962,0.0,0.0,0.0,0.0,1.0,0.299949
weekday,4472500.0,3.003354,0.0,1.0,3.0,5.0,6.0,1.999718
month,4472500.0,6.594746,1.0,4.0,7.0,10.0,12.0,3.391799
year,4472500.0,2021.025154,2019.0,2020.0,2021.0,2022.0,2023.0,1.399087
quarter,4472500.0,2.528228,1.0,2.0,3.0,4.0,4.0,1.107378
week,4472500.0,26.915595,1.0,14.0,27.0,40.0,53.0,14.811284
day,4472500.0,15.683622,1.0,8.0,16.0,23.0,31.0,8.784386


Columnas finales:
['date', 'store_id', 'item_id', 'sales', 'price', 'promo', 'weekday', 'month', 'year', 'quarter', 'week', 'day', 'sales_lag_7', 'sales_lag_14', 'sales_lag_30', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_std_30', 'trend_7_30', 'stock_estimated', 'safety_stock', 'lead_time_days', 'future_demand_7d', 'stockout_risk']


## 14. Guardado del dataset procesado

Se guarda el dataset procesado en la carpeta `data/processed`.

Este fichero será la entrada del notebook de entrenamiento de modelos.

In [14]:
df.to_csv(
    PROCESSED_DATA_PATH,
    index=False
)

print(f"Dataset procesado guardado correctamente en: {PROCESSED_DATA_PATH}")

Dataset procesado guardado correctamente en: ../data/processed/stockout_dataset.csv


## 15. Conclusiones de ingeniería de variables

En este notebook se han creado variables temporales, variables de demanda histórica y variables logísticas simuladas para construir un problema de clasificación binaria orientado a la predicción de riesgo de rotura de stock.

Las variables más relevantes para el modelado posterior son las relacionadas con:

- Comportamiento histórico de ventas.
- Tendencias recientes de demanda.
- Estacionalidad temporal.
- Precio y promociones.
- Lead time simulado.

También se han identificado variables que deben excluirse del entrenamiento para evitar leakage. Esta separación es clave para obtener una evaluación más realista del rendimiento del modelo.